# Lesson 2 · Phase 2 — Neurons, Weights & Activations → Predictions

**Mastering Agentic AI Certification · Pre-read**

> Phase 1 turned text into **embedding vectors**. Now we feed those vectors through the network. This phase shows what a **neuron**, a **weight**, and an **activation** actually are, and how stacking them takes an **Input → Layer → Prediction**.

## 1. The complete picture (the forward pass)

```
  embedding x        weights W, bias b        nonlinearity        next layer / output
 [0.9,0.1,0.2,0] ─▶  z = W·x + b        ─▶   a = activation(z) ─▶  ... ─▶ logits ─▶ softmax ─▶ P(next token)

         INPUT      ───────────── one LAYER ─────────────                       PREDICTION
```

Running data **forward** through these steps to get an output is called the **forward pass**.

## 2. A single neuron

A neuron does two tiny things:

1. **Weighted sum** of its inputs plus a bias:  $z = w_1x_1 + w_2x_2 + \dots + w_nx_n + b = \mathbf{w}\cdot\mathbf{x} + b$
2. **Activation:**  $a = \sigma(z)$  — a nonlinear function applied to $z$.

That's it. A neuron is "multiply-add, then bend."

## 3. Weights & biases — the learnable parameters

- **Weights** decide *how much each input matters* (large weight ⇒ that input strongly influences the neuron).
- **Bias** shifts the neuron's threshold up/down.
- Together, **weights + biases are "the parameters"** — the billions of numbers that get **learned** in training (Phase 3). The embeddings from Phase 1 are parameters too.

> Learning = finding the weight values that make good predictions.

## 4. Activations — why nonlinearity is essential

Without a nonlinear activation, stacking layers is pointless: a chain of linear steps collapses into **one** linear step, so the model could only draw straight lines.

The activation **bends** the signal so the network can represent curves, logic, and complex patterns. Common ones:

| Activation | Formula | Note |
|---|---|---|
| **ReLU** | $\max(0, z)$ | Cheap, default in most deep nets; no saturation for $z>0$ |
| **Sigmoid** | $1/(1+e^{-z})$ | Squashes to (0,1); *saturates* → causes vanishing gradients (Phase 3) |
| **Tanh** | $\tanh(z)$ | Squashes to (−1,1); also saturates |
| **GELU/SiLU** | smooth ReLU-like | Used in modern Transformers |

## 5. A layer = many neurons at once (one matrix multiply)

Stacking $h$ neurons that all see the same input $\mathbf{x}$ lets us batch the math into a **matrix multiply**:

$$\mathbf{z} = W\mathbf{x} + \mathbf{b}, \qquad \mathbf{a} = \text{activation}(\mathbf{z})$$

where $W$ has one **row of weights per neuron**. This is why GPUs (built for matrix multiplies) are perfect for neural nets — and it's the same operation the Transformer's feed-forward blocks run.

In [ ]:
import numpy as np
np.random.seed(1)

# INPUT: an embedding vector from Phase 1 (here d_in = 4)
x = np.array([0.9, 0.1, 0.2, 0.0])

def relu(z):    return np.maximum(0, z)
def softmax(z): e = np.exp(z - z.max()); return e / e.sum()

# ---- Hidden LAYER: 5 neurons. W1 has one ROW of weights per neuron ----
W1 = np.random.randn(5, 4) * 0.7      # (neurons=5, inputs=4)
b1 = np.zeros(5)
z1 = W1 @ x + b1                       # 1. weighted sum  (per neuron)
a1 = relu(z1)                          # 2. activation

# ---- OUTPUT LAYER: scores over a tiny 3-token vocabulary ----
vocab = ["sleep", "run", "rocks"]
W2 = np.random.randn(3, 5) * 0.7
b2 = np.zeros(3)
logits = W2 @ a1 + b2                  # raw scores (one per possible next token)
probs  = softmax(logits)              # PREDICTION: a probability distribution

print("INPUT embedding x      :", x)
print("hidden pre-activation z1:", np.round(z1, 2))
print("hidden activation a1   :", np.round(a1, 2), " <- ReLU zeroed the negatives")
print("logits                 :", np.round(logits, 2))
print("\nPREDICTED P(next token):")
for w, p in sorted(zip(vocab, probs), key=lambda t: -t[1]):
    print(f"  {w:<6} {p:6.1%}")
print("=> next-token prediction:", vocab[int(np.argmax(probs))])

## 6. Input → Layer → Prediction (and stacking depth)

You just ran the full chain for one layer + an output head:

```
 x ─▶ [W1·x+b1 → ReLU] ─▶ a1 ─▶ [W2·a1+b2] ─▶ logits ─▶ softmax ─▶ P(next token)
INPUT        LAYER 1          (hidden)     OUTPUT LAYER            PREDICTION
```

Real LLMs simply **stack many such layers** (the output of one becomes the input of the next). Depth lets early layers detect simple patterns and later layers combine them into abstract concepts — but stacking also creates the **gradient problems** of Phase 3.

In [ ]:
# COMPLETE PICTURE: visualise the forward pass as activations flowing layer to layer.
import numpy as np, matplotlib.pyplot as plt
layers = [("input x", x), ("hidden a1 (ReLU)", a1), ("logits", logits), ("probs", probs)]
fig, axes = plt.subplots(1, len(layers), figsize=(11, 3))
for ax, (name, v) in zip(axes, layers):
    ax.bar(range(len(v)), v, color="steelblue")
    ax.set_title(name, fontsize=10); ax.axhline(0, color="grey", lw=0.6)
fig.suptitle("Forward pass: Input → Layer → Prediction (values per unit)", y=1.05)
plt.tight_layout(); plt.show()

## 7. How this contributes

This forward pass is exactly **where the next-token distribution from Lesson 1 comes from**:

- The **embeddings** (Phase 1) are the input `x`.
- **Weights & biases** are the parameters pre-training/fine-tuning adjust (Lessons 1–2).
- The final **softmax over logits** *is* the `P(next token | context)` the model is trained to get right.

```
embeddings ─▶ (neurons · weights · activations across many layers) ─▶ logits ─▶ softmax ─▶ prediction
```

But a freshly-initialised network predicts nonsense — its weights are random. **Phase 3** shows how gradients and backprop fix that.

## 8. Key takeaways

1. A **neuron** = weighted sum (`W·x + b`) followed by a **nonlinear activation**.
2. **Weights & biases are the learnable parameters**; **activations add the nonlinearity** that makes depth useful.
3. A **layer** is many neurons = one **matrix multiply** (why GPUs shine).
4. **Input → Layer(s) → logits → softmax → prediction** is the **forward pass**.
5. This forward pass produces Lesson 1's **next-token probability**, but only becomes accurate after **learning** (Phase 3).

---
*Next (Phase 3):* computing **gradients**, **backpropagation**, and the **failure modes** (vanishing & exploding gradients).

In [ ]:
import sys, platform
print("Python :", sys.version.split()[0]); print("Platform:", platform.platform())
print("Lesson 2 · Phase 2 (Neurons → Predictions) notebook is running ✓")